In [ ]:
%pip install sb3-contrib

In [ ]:


"""
key_wards_eval["runners"] = [
        (lambda: gym_agents.RunnerEscape(vel_seq_len=vel_seq_len, energy=1000, speed=0.5), 1),
        (lambda: gym_agents.RunnerCircle(vel_seq_len=vel_seq_len, energy=1000, speed=0.5, radius=50.0), 1),
        (lambda: gym_agents.RunnerRandom(vel_seq_len=vel_seq_len, energy=-1000, speed=0.5), 1),
        (lambda: gym_agents.RunnerEscapeWhenTooClose(vel_seq_len=vel_seq_len, energy=1000, threshold=0.2, speed=0.5), 1),
        (lambda: gym_agents.RunnerStop(vel_seq_len=vel_seq_len, energy=1000), 1),
        (lambda: gym_agents.RunnerStopAndMove(speed=0.5, move_interval=10, vel_seq_len=vel_seq_len, energy=1000), 1),
        (lambda: gym_agents.RunnerSnake(vel_seq_len=vel_seq_len, energy=1000, speed=0.5, amplitude=1.0, freq=1.0), 1),
        (lambda: gym_agents.RunnerOscillation(vel_seq_len=vel_seq_len, energy=1000, axis='x', amplitude=10.0, freq=1/10), 1),
        (lambda: gym_agents.Enemy(vel_seq_len=vel_seq_len, energy=1000, speed=0.5), 1)
]
"""


In [ ]:
import os
import json
from stable_baselines3 import DQN

import cat_toy_env
import gym_agents
import cat_actions

import importlib
importlib.reload(cat_toy_env)
importlib.reload(gym_agents)
importlib.reload(cat_actions)

# モデルIDを指定（例: '1'）
model_id = '1'

def get_config_path(model_id):
    # 設定ファイルのパスを生成
    return os.path.join('models', model_id, 'config.json')

def load_env_and_config(model_id):
    config_path = get_config_path(model_id)
    with open(config_path) as f:
        config = json.load(f)

    env_config = config['env']['kwargs']

    # chaser（Cat）
    chaser_class = gym_agents.gym_agents_mapping[config['chaser']['name']]
    chaser_kwargs = config['chaser'].get('kwargs', {}).copy()
    # cat_actionsの文字列リストをクラスインスタンスに変換
    if 'cat_actions' in chaser_kwargs:
        chaser_kwargs['cat_actions'] = [cat_actions.cat_actions_mapping[name]() for name in chaser_kwargs['cat_actions']]
    chaser = lambda: chaser_class(**chaser_kwargs)

    # runners
    runners = []
    for runner_name, runner_info in config['runners'].items():
        runner_class = gym_agents.gym_agents_mapping[runner_name]
        runner_kwargs = runner_info['kwargs']
        runner_kwargs['vel_seq_len'] = config['env']['vel_seq_len']  # config.jsonの値で上書き
        probability = runner_info.get('probability', 1.0)
        runners.append((lambda rc=runner_class, rk=runner_kwargs: rc(**rk), probability))

    # 環境構築
    key_wards = {
        **env_config,
        'chaser': chaser,
        'runners': runners,
    }
    env = cat_toy_env.CatToyEnv(**key_wards)
    return env, key_wards, config

env, key_wards, config = load_env_and_config(model_id)
model = DQN("MlpPolicy", env, verbose=1)
total_timesteps = 0

## モデルをロード（必要な場合のみ）

In [ ]:
model = DQN.load(f"models/{model_id}/cat")
env, key_wards, config = load_env_and_config(model_id)
model.set_env(env)  # 必要に応じて新しい環境をセット

if 'total_timesteps' in config:
    total_timesteps = config['total_timesteps']
else:
    total_timesteps = 0

## 学習

In [ ]:
current_learning_steps = 150_000
# 学習
model.learn(total_timesteps=current_learning_steps, log_interval=4)
model.save(f'models/{model_id}/cat')

# 合計timestepsをconfig.jsonに保存
total_timesteps += current_learning_steps
config['total_timesteps'] = total_timesteps
with open(get_config_path(model_id), 'w') as f:
    json.dump(config, f, indent=2)

In [ ]:
import numpy as np

key_wards_eval = key_wards.copy()
key_wards_eval["render_mode"] = "human"

env_eval = cat_toy_env.CatToyEnv(**key_wards_eval)

model = DQN.load(f"models/{model_id}/cat")

obs, _ = env_eval.reset()
# cell and hidden state of the LSTM
lstm_states = None
num_envs = 1
# Episode start signals are used to reset the lstm states
episode_starts = np.ones((num_envs,), dtype=bool)
done = False
while not done:
    action, lstm_states = model.predict(obs, deterministic=True)
    obs, rewards, terminated, truncated, info = env_eval.step(action)
    done = terminated or truncated

In [ ]:
import torch
import dqn_onnx
importlib.reload(dqn_onnx)

policy_net = dqn_onnx.DQNOnnx(model.policy)
env_dummy = cat_toy_env.CatToyEnv(**key_wards_eval)
obs_dim = env_dummy.observation_space.shape[0]
obs = torch.randn(1, obs_dim).detach().cpu()

torch.onnx.export(
    policy_net,
    (obs),
    f'models/{model_id}/policy.onnx',
    export_params=True,
    opset_version=17,
    input_names=['obs'],
    output_names=['option', 'action'],
    dynamic_axes={
        'obs': {0: 'batch_size'},
        'option': {0: 'batch_size'},
        'action': {0: 'batch_size'},
    },
    training=torch.onnx.TrainingMode.EVAL
)

In [ ]:
%mkdir -p ../cat-game/public/models/{model_id}
%cp models/{model_id}/policy.onnx ../cat-game/public/models/{model_id}/policy.onnx
%cp models/{model_id}/config.json ../cat-game/public/models/{model_id}/config.json